In [1]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, TextLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

In [2]:
DATA_DIR = Path("../data")
VECTORSTORE_DIR = Path("../vectorstore/chroma_db")

loader = DirectoryLoader(
    str(DATA_DIR),
    glob="**/*.txt",
    loader_cls=TextLoader,
    show_progress=True
)

documents = loader.load()
print(f"Loaded {len(documents)} pages from {len(set(d.metadata['source'] for d in documents))} documents")

100%|██████████| 1/1 [00:00<00:00, 1404.66it/s]

Loaded 1 pages from 1 documents


In [3]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks from {len(documents)} pages")
print(f"\nSample chunk:\n{chunks[0].page_content[:300]}")

Created 12 chunks from 1 pages

Sample chunk:
CMS Proposes Major Reforms to Speed Up Patient Access to Drugs, Increase Transparency, and Reduce Administrative Burden
Administration
Share

CMS Proposes Major Reforms to Speed Up Patient Access to Drugs, Increase Transparency, and Reduce Administrative Burden 

Proposed rule would require faster p


In [4]:
print("Loading embedding model (first run downloads ~90MB)...")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=str(VECTORSTORE_DIR)
)

vectorstore.persist()
print(f"Stored {vectorstore._collection.count()} chunks in Chroma")

Loading embedding model (first run downloads ~90MB)...


/Users/prasunamannava/clinical-prior-auth-assistant/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/prasunamannava/clinical-prior-auth-assistant/venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Stored 12 chunks in Chroma


/Users/prasunamannava/clinical-prior-auth-assistant/venv/lib/python3.11/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  warn_deprecated(


In [5]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

query = "What documentation is required to support a prior auth request for an MRI?"

results = retriever.get_relevant_documents(query)

print(f"Query: {query}\n")
print("=" * 60)
for i, doc in enumerate(results, 1):
    print(f"\nResult {i}")
    print(f"Source: {doc.metadata.get('source', 'unknown')}")
    print(f"\n{doc.page_content}")
    print("-" * 60)

/Users/prasunamannava/clinical-prior-auth-assistant/venv/lib/python3.11/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Query: What documentation is required to support a prior auth request for an MRI?


Result 1
Source: ../data/cms_prior_auth_overview.txt

Building on CMS’ 2024 Interoperability and Prior Authorization final rule, which addressed prior authorization for non-drug items and services, this proposal aims to ensure patients experience the same streamlined process for medications as other covered services.
------------------------------------------------------------

Result 2
Source: ../data/cms_prior_auth_overview.txt

“Last year, we got 80 percent of the insurance industry to agree to eliminate prior authorization for common medical services such as diagnostic imaging, physical therapy, and outpatient surgery,” Health and Human Services Secretary Kennedy said. “This rule builds on that agreement by making it easier for patients to get the medications they need by minimizing delays and enabling real-time decisions.”
------------------------------------------------------------

Result 3
Sourc

In [6]:
vectorstore_loaded = Chroma(
    persist_directory=str(VECTORSTORE_DIR),
    embedding_function=embeddings
)

print(f"Reloaded vectorstore with {vectorstore_loaded._collection.count()} chunks")
print("Persistence confirmed — Sprint 1 complete.")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Reloaded vectorstore with 12 chunks
Persistence confirmed — Sprint 1 complete.
